In the bronze layer of this pipeline - we will be ingesting data for the 4 major economic indicators:
Unemployment
GDP (adjusted for inflation)
Inflation (in terms of CPI- Consumer Price Index)
Total Trade Balance

In [65]:
import requests
import pandas as pd

In [ ]:
def get_series_uri(cdid:str) -> str:
    resp = requests.get(
         "https://api.beta.ons.gov.uk/v1/search",
        params={"content_type": "timeseries", "cdids": cdid},
        timeout=30,
    )
    resp.raise_for_status()
    items = resp.json() ["items"]
    return items[0]["uri"]

def fetch_series(uri: str) -> dict:
    resp = requests.get(
        "https://api.beta.ons.gov.uk/v1/data",
        params={"uri": uri},
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()




# Fetching the data from ONS
To fetch the different data from the ONS API - we need their CDID and their Dataset - creating a function that only requires the different CDIDs and datasets make fetching of data much more easier.


## Unemployment Rate 
CDID: MGSX
Dataset: LMS

## CPI Inflation
CDID: D7G7
Dataset: MM23

## GDP
CDID: ABMI
Dataset: PN2

## Total Trade Balance
CDID: IKBJ
Dataset: MRET




In [74]:
unemployment_uri = get_series_uri("MGSX")
unemployment_data = fetch_series(unemployment_uri)
print(unemployment_data.keys())
rows = []
for obs in unemployment_data['quarters']:
    rows.append({
        "date": obs['date'],
        "value": obs['value'],
    })
unemployment_df = pd.DataFrame(rows)
display(unemployment_df)

dict_keys(['years', 'quarters', 'months', 'sourceDatasets', 'relatedDatasets', 'section', 'relatedDocuments', 'relatedData', 'alerts', 'versions', 'type', 'uri', 'description'])


,date,value
0,1971 Q1,3.8
1,1971 Q2,4.1
2,1971 Q3,4.2
3,1971 Q4,4.4
4,1972 Q1,4.5
...,...,...
217,2025 Q2,4.7
218,2025 Q3,5.0
219,2025 Q4,5.2
220,2026 Q1,5.0


In [71]:
get_series_uri('D7G7')
inflation_data = fetch_series('/economy/inflationandpriceindices/timeseries/d7g7/mm23')
rows = []
for obs in inflation_data['quarters']:
    rows.append({
        'date': obs['date'],
        'value': obs['value'],
    }
    )
inflation_df = pd.DataFrame(rows)
display(inflation_df)

,date,value
0,1989 Q1,5.0
1,1989 Q2,5.3
2,1989 Q3,5.1
3,1989 Q4,5.5
4,1990 Q1,5.8
...,...,...
145,2025 Q2,3.5
146,2025 Q3,3.8
147,2025 Q4,3.4
148,2026 Q1,3.1


In [72]:
get_series_uri('ABMI')
gdp_data = fetch_series('/economy/grossdomesticproductgdp/timeseries/abmi/pn2')
rows = []
for obs in gdp_data['quarters']:
    rows.append({
        'date': obs['date'],
        'value' : obs['value']
    })
gdp_df = pd.DataFrame(rows)
display(gdp_df)


,date,value
0,1955 Q1,145457
1,1955 Q2,145551
2,1955 Q3,147995
3,1955 Q4,147119
4,1956 Q1,148955
...,...,...
281,2025 Q2,704274
282,2025 Q3,704802
283,2025 Q4,705160
284,2026 Q1,709598


In [73]:
get_series_uri('IKBJ')
trade_balance_data = fetch_series('/economy/nationalaccounts/balanceofpayments/timeseries/ikbj/mret')
rows = []
for obs in trade_balance_data['quarters']:
    rows.append({
        'date': obs['date'],
        'value': obs['value']
    })
trade_balance_df = pd.DataFrame(rows)
display(trade_balance_df)

,date,value
0,1955 Q1,
1,1955 Q2,
2,1955 Q3,
3,1955 Q4,
4,1956 Q1,
...,...,...
281,2025 Q2,-11284
282,2025 Q3,-7685
283,2025 Q4,-11468
284,2026 Q1,-14726


In [54]:
import duckdb

con = duckdb.connect("econ.duckdb")

In [ ]:
con.execute("CREATE OR REPLACE TABLE unemployment AS SELECT * FROM unemployment_df")
con.execute("CREATE OR REPLACE TABLE inflation AS SELECT * FROM inflation_df")
con.execute("CREATE OR REPLACE TABLE GDP AS SELECT * FROM gdp_df")
con.execute("CREATE OR REPLACE TABLE trade_balance AS SELECT * FROM trade_balance_df")

In [77]:
con.close()